In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet("data/pq/green/*/*")

In [ ]:
#si crea una vista temporanea per poter eseguire query sql
df_green.createOrReplaceTempView ("green")

In [ ]:
df_green_revenue = spark.sql("""
SELECT
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount, 
    COUNT(1) AS number_records
FROM green
WHERE  lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1,2
ORDER BY 1,2
""").show()

In [ ]:
df_green_revenue.write.parquet("data/report/revenue/green")

Parte per il Dataset YELLOW 
(stessa cosa che è stata fatta con green)

In [ ]:
df_yellow = spark.read.parquet("data/pq/yellow/*/*")

In [ ]:
#si crea una vista temporanea per poter eseguire query sql
df_yellow.createOrReplaceTempView ("yellow")

In [ ]:
df_yellow_revenue = spark.sql("""
SELECT
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    SUM(total_amount) AS amount, 
    COUNT(1) AS number_records
FROM yellow
WHERE  tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY 1,2
ORDER BY 1,2
""").show()

In [ ]:
df_yellow_revenue.write.parquet("data/report/revenue/yellow")

Parte sulle Joins

Quello che si fa in questa parte è la Join tra la tabella Green e Yellow. 
Quindi si fa tipo la Union vista in precedenza e anche sulla parte di dbt

In [ ]:
df_join = df_green_revenue.join(df_yellow_revenue, on=["hour", "zone"], how="outer")

NOTA: in on=[] si specificano le colonne su cui avviene l'operazione di Join, in questo caso le 2 colonne che sono chiavi. 

how = 'outer' -> prende tutti i record che sono in green ma non in yellow. 

In [ ]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed("amount", "green_amount") \
    .withColumnRenamed("number_records", "green_number_records")

In [ ]:
df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed("amount", "yellow_amount") \
    .withColumnRenamed("number_records", "yellow_number_records")

In [ ]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=["hour", "zone"], how="outer")

In [ ]:
df_join.write.parquet("data/report/revenue/total")

Adesso vediamo cosa succede quando bisogna fare la Join tra una tabella molto grande e una piccola.

NOTA: si fa l'operazione di Broadcast tra i 2 Dataset (si può vedere su Spark UI). 
Ovvero che il Dataset più piccolo viene copaito in broadcast dentro tutti gli Executors e la join avviene in-memory dentro ogni Executor, senza fare l'operazione di Shuffle dei dati tra i vari Executors. 

In [ ]:
# Dataset "grande"
df_join = spark.read.parquet("data/report/revenue/total")

In [ ]:
#Dataset "piccolo" 
df_zones = spark.read.csv("taxi_zone_lookup.csv", header=True)

In [ ]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [ ]:
df_result = df_result.drop("zone")

In [ ]:
df_result.write.parquet("tmp/revenue-zones")